# Apply Work-Author Curations

Applies `add` and `remove` work-author curations from `openalex.works.work_author_{add,remove}_curations` onto `openalex.works.work_authors`. Runs after `Author_Matching` and before `UpdateWorkAuthorships`.

## Apply-once semantics

Each curation row carries an `applied_at TIMESTAMP` column. It's NULL when first synced; this notebook stages everything where `applied_at IS NULL`, MERGEs into `work_authors`, queues affected work_ids for re-sync, then stamps `applied_at = current_timestamp()` to mark them done. Subsequent runs filter past anything already stamped — no re-application, no risk of misapplying after upstream re-ingest reorders the authorship array.

Why apply-once is safe (verified 2026-05-08): nothing in the pipeline currently re-orphans a curated slot.
- `MatchAuthors` only writes to slots where `author_id IS NULL` (its `needs_matching` CTE filters on `wa.author_id IS NULL`).
- `UpdateWorkAuthors` on MATCHED rows updates `raw_author_name`, `raw_affiliation_strings`, `is_corresponding`, `updated_at` — explicitly NOT `author_id`.

So a curated `author_id`, once written, persists through subsequent cycles without our help.

## Semantics

- **`add(B, W, k)` = transfer**: positional MERGE on `(work_id, author_sequence)`. No name guard — `raw_author_name` on the source row is frozen at sync time as a diagnostic anchor only (audit trail of what the curator saw).
- **`remove(A, W)` = sticky disclaim**: MERGE on `(work_id, author_id)` NULLs `author_id` at any matching slot. Block-list filter in `MatchAuthors.blocked_candidates` (reading `work_author_remove_curations` directly) keeps A from being re-selected on subsequent cycles.

## Affected works queue

Only newly-applied work_ids (this run's pending batch) get inserted into `openalex.institutions.curated_work_ids_pending_sync`, so `UpdateWorkAuthorships` doesn't re-flow already-applied curations every cycle.

## Stage pending curations

Snapshot what's unapplied at the start of this run. Subsequent steps drive off these tables, so a row added to the source between staging and stamping won't get prematurely marked applied.

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.works.work_author_curation_pending_add AS
SELECT *
FROM openalex.works.work_author_add_curations
WHERE applied_at IS NULL

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.works.work_author_curation_pending_remove AS
SELECT *
FROM openalex.works.work_author_remove_curations
WHERE applied_at IS NULL

## Apply add curations to work_authors

Positional MERGE on `(work_id, author_sequence)` from the pending batch. Skips the UPDATE when the slot already holds the curated author (no-op churn protection — useful when an existing applied row gets re-staged because its `applied_at` was cleared manually).

In [ ]:
%sql
MERGE INTO openalex.works.work_authors AS target
USING openalex.works.work_author_curation_pending_add AS source
ON target.work_id = source.work_id
   AND target.author_sequence = source.author_sequence
WHEN MATCHED AND (target.author_id IS NULL OR target.author_id <> source.author_id) THEN
  UPDATE SET
    target.author_id  = source.author_id,
    target.updated_at = current_timestamp()

## Apply remove curations to work_authors

NULL the `author_id` at any (work_id, author_sequence) where it currently equals the disclaimed author.

In [ ]:
%sql
MERGE INTO openalex.works.work_authors AS target
USING openalex.works.work_author_curation_pending_remove AS source
ON target.work_id = source.work_id
   AND target.author_id = source.author_id
WHEN MATCHED THEN
  UPDATE SET
    target.author_id  = NULL,
    target.updated_at = current_timestamp()

## Queue affected work_ids for re-sync

Insert only newly-applied work_ids (this run's batch) into `curated_work_ids_pending_sync`. `UpdateWorkAuthorships` picks them up and truncates the queue at the end of its run.

In [ ]:
%sql
INSERT INTO openalex.institutions.curated_work_ids_pending_sync
SELECT DISTINCT work_id, NULL AS curated_ras, current_timestamp() AS added_datetime
FROM (
    SELECT work_id FROM openalex.works.work_author_curation_pending_add
    UNION
    SELECT work_id FROM openalex.works.work_author_curation_pending_remove
)

## Mark applied

Stamp `applied_at` on every row in this run's pending batches. Driven off the snapshot tables, not `applied_at IS NULL` directly, so any row added to the source between staging and now is preserved for the next run.

In [ ]:
%sql
UPDATE openalex.works.work_author_add_curations
SET applied_at = current_timestamp()
WHERE curation_id IN (SELECT curation_id FROM openalex.works.work_author_curation_pending_add)

In [ ]:
%sql
UPDATE openalex.works.work_author_remove_curations
SET applied_at = current_timestamp()
WHERE curation_id IN (SELECT curation_id FROM openalex.works.work_author_curation_pending_remove)

## Verify

In [ ]:
%sql
SELECT
    (SELECT COUNT(*) FROM openalex.works.work_author_curation_pending_add)               AS adds_applied_this_run,
    (SELECT COUNT(*) FROM openalex.works.work_author_curation_pending_remove)            AS removes_applied_this_run,
    (SELECT COUNT(*) FROM openalex.works.work_author_add_curations)                      AS add_curations_total,
    (SELECT COUNT(*) FROM openalex.works.work_author_add_curations WHERE applied_at IS NOT NULL)
                                                                                         AS add_curations_applied_total,
    (SELECT COUNT(*) FROM openalex.works.work_author_remove_curations)                   AS remove_curations_total,
    (SELECT COUNT(*) FROM openalex.works.work_author_remove_curations WHERE applied_at IS NOT NULL)
                                                                                         AS remove_curations_applied_total

In [ ]:
%sql
-- Spot-check work_authors for everything touched by curations (not just this run).
-- Useful for confirming applied curations are still in place across cycles.
SELECT wa.work_id, wa.author_sequence, wa.author_id, wa.raw_author_name, wa.updated_at
FROM openalex.works.work_authors wa
WHERE wa.work_id IN (
    SELECT work_id FROM openalex.works.work_author_add_curations
    UNION
    SELECT work_id FROM openalex.works.work_author_remove_curations
)
ORDER BY wa.updated_at DESC
LIMIT 100